# Section 1. Introduction to LangChain

## Introduction

LangChain is an open-source framework that simplifies the development of applications powered by language models (LLMs). In a rapidly evolving landscape of language models, LangChain stands out by offering a more accessible and flexible way to integrate these technologies into a wide range of applications. This enables the creation of smarter and more personalized solutions, tailored to the specific needs of each context.

LangChain applications span various areas:
* Intelligent and conversational chatbots;
* Creative multimedia content generation;
* Data analysis for extracting valuable insights;
* Virtual assistants capable of automating complex tasks;
* Accurate and informative question-and-answer systems.

## 1.1 About API

Unlike direct interaction with language models through interfaces like ChatGPT, LangChain uses APIs (Application Programming Interfaces) for automated communication. These APIs act as bridges, allowing LangChain to connect to and interact with LLMs via web protocols.

Each LLM provider offers its own API, and to use it, you need to obtain an access key. This key serves as authentication, enabling the service to identify and authorize your requests, execute them, and return the appropriate responses.

The OpenAI API is widely used, known for its robustness and low error rate. However, it is a paid service and can become expensive depending on the model used. For learning and testing purposes, we recommend Google's API, which offers a generous amount of free requests. Check the usage limits at this [link](https://ai.google.dev/gemini-api/docs/rate-limits?hl=pt-br).

To create your account and obtain access keys, visit the links below. Remember to store your keys in a safe place, as they won't be displayed again and losing them will require generating new ones.

* Gemini (Google): https://ai.google.dev/gemini-api/docs/api-key
* OpenAI: https://platform.openai.com


Now let's start using LangChain!
First, we need to install the necessary dependencies to run LangChain with Google's Gemini model by executing the cell below.

In [1]:
!pip install -qU python-dotenv langchain langchain-openai langchain-google-genai langchain-classic langchain-openrouter openrouter

One way to save your keys is by using a `.env` file. If you wish, create a file named `.env` and save your keys as follows:
```
GOOGLE_API_KEY = …
OPENAI_API_KEY = …
OPENROUTER_API_KEY = …
LANGSMITH_API_KEY = …
LANGSMITH_TRACING = false
```

In [2]:
import getpass
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

## 1.2 Inicialização do modelo de linguagem

The function below simplifies the use of different models (Gemini, OpenAI or OpenRouter). You can select which model you want to use and also adjust some parameters.

**In current LangChain versions, `init_chat_model(...)` is the recommended way to initialize chat models while keeping a unified interface across providers.** This makes it easier to switch providers without changing the rest of the notebook.

#### Temperature:
Temperature controls how confident the model needs to be before choosing the next word.

* High Temperature (>0.7):
  * Increases randomness. The model is more likely to pick less probable words, leading to more creative, surprising, or even chaotic responses.
  * Good for storytelling, brainstorming, or when you want variety.

* Low Temperature (<0.5):
  * Makes output more predictable and focused. The model sticks to high-probability choices.
  * Ideal for factual, logical, or structured tasks.
* 0.7 is a common default for chatbots.
* 0.0 is a common default for agents.

#### Top-P (Nucleus Sampling):
Top-p filters possible next words by cumulative probability ensuring coherent diversity. Top-p functions by dynamically filtering based on context, create a more natural and fluent output.

The model sorts all possible next words by likelihood, then only considers the top ones whose total probability adds up to p (e.g., 0.9).
  * This means the model won't just take the top 10, but rather the top words that collectively feel "safe enough."

* 0.8-0.95 for balanced diversity and relevance.
* Lower values (like 0.6) make the output more narrow and safe.


You can also test these parameters on [Google AI Studio](https://aistudio.google.com/prompts/new_chat).

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_openrouter import ChatOpenRouter

def get_model_name(model_name, temperature=0):
    if model_name == "gemini": # https://ai.google.dev/gemini-api/docs/rate-limits?hl=pt-br
        if "GOOGLE_API_KEY" not in os.environ: # https://ai.google.dev/gemini-api/docs/api-key
            os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")
        llm = ChatGoogleGenerativeAI(
            # model="gemini-1.5-pro", # max 50 / dia
            model="gemini-1.5-flash", # max 1500 / dia
            temperature=temperature,
        )
    elif model_name == "openai":
        if "OPENAI_API_KEY" not in os.environ: # https://platform.openai.com
            os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=temperature,
        )
    elif model_name == "openrouter": # https://openrouter.ai/workspaces/default/
        if not os.getenv("OPENROUTER_API_KEY"):
            os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

        llm = ChatOpenRouter(
            model="openrouter/free",
            api_key=os.environ["OPENROUTER_API_KEY"],
            temperature=temperature,
        )

    return llm

llm = get_model_name('openrouter')

Now that we have a `BaseChatModel` object named `llm`, we can test it by making our first API call using the `invoke` method with a given prompt. Try changing the prompt to observe different outputs.

In [4]:
prompt = "Hello, how are you?"
response = llm.invoke(prompt)
print(response)

content="I am doing well, thank you for asking! As a large language model, I don't experience feelings like humans do, but I'm functioning optimally and ready to assist you. 😊 \n\nHow are *you* doing today? I hope you're having a good one! Is there anything I can help you with?\n\n\n\n" additional_kwargs={} response_metadata={'model_name': 'google/gemma-3n-e4b-it:free', 'id': 'gen-1776281477-3v4H0WhSnN324mwydYFS', 'created': 1776281477, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter'} id='lc_run--019d92a0-5e89-7753-ae54-5bdeb5fac8c3-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 7, 'output_tokens': 68, 'total_tokens': 75, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}


Another way to interact with the API is by setting up a conversation with the AI. The cell below creates a list called `messages`. **In current LangChain documentation, messages are treated as the fundamental unit of context for chat models.** This list is used to structure the conversation that will be sent to the language model (LLM).

* Each element in the `messages` list is a tuple containing two strings.
* The first string indicates the role of the message (`"system"` or `"human"`).
* The second string contains the content of the message.
* `("system", ...)`: Defines a system message. System messages provide instructions or context to the LLM, guiding its behavior. In this case, you're instructing the LLM to act as an assistant that translates English to French.
* `("human", ...)`: Defines a user (human) message. This is the user's input that the LLM will process. In this case, the input is the sentence "I love programming."

**This tuple format is a convenient shorthand; LangChain also supports explicit message objects such as `SystemMessage`, `HumanMessage`, and `AIMessage`.**

In [5]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
response = llm.invoke(messages)
print(response.content)

J'adore programmer.


## 1.3 Prompt Template

A fundamental technique when using LangChain is the application of Prompt Templates. They allow developers to create dynamic and reusable prompts, where variables can be inserted to customize the instructions sent to LLMs during workflow execution. This not only simplifies the creation of complex prompts but also ensures consistency and makes application maintenance easier.

The syntax is simple, as shown in the cell below: you define a string using triple double-quotes, and variables are inserted between curly braces. These variables represent the text that will be dynamically modified within the template.

In [6]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [9]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)

In [10]:
prompt_template.messages[0].prompt

PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n')

Here we can see that `prompt_template` takes two input variables: `['style', 'text']`.










In [11]:
prompt_template.messages[0].prompt.input_variables

['style', 'text']

Now, let’s input the two variables and build the prompt that we’ll send to the LLM.

In [12]:
customer_style = """American English \
in a calm and respectful tone
"""

customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

customer_messages = prompt_template.format_messages(
                    style=customer_style,
                    text=customer_email)

print(type(customer_messages))
print(type(customer_messages[0]))
print(customer_messages[0])

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>
content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone\n. text: ```\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n" additional_kwargs={} response_metadata={}


After using the `format_messages` method with the input variables, we build the final prompt. When inspecting `customer_messages`, we see that it consists of a list containing a single element of type `HumanMessage`. In LangChain, `HumanMessage` represents the prompts that the LLM should respond to, and not necessarily a message written by a human, as we’ll see later.

In [13]:
# Call the LLM to translate to the style of the customer message
customer_response = llm.invoke(customer_messages)
print(customer_response.content)

"I'm quite upset that my blender lid came off and splattered my kitchen walls with smoothie. To make matters worse, the warranty doesn't cover the cost of cleaning up my kitchen. I could really use your help right now."


## 1.3 Output Parsers

Now that we’ve learned how to structure the message we send to the LLM, let’s learn how to ask the LLM to return in a structured format using Output Parsers. They allow developers to define the desired format for the output, whether it's JSON, lists, objects, or any other specific structure. This is especially useful in applications that require extracting precise and structured information from LLMs, making it easier to integrate with other tools and systems.

First, let's start with defining how we would like the LLM output to look like:

```python
{
  "gift": False,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
}
```

In LangChain, an effective way to define Output Parsers is by using the `pydantic` library. Pydantic is a Python library that allows the creation of robust data models using type annotations, offering data validation and automatic serialization/deserialization. For this purpose, we import:

* `BaseModel`: the base class for defining structured data models.
* `Field`: used to define individual fields within the model, allowing the addition of descriptive metadata.

We can define the desired output structure by creating a class that inherits from `BaseModel`. Inside this class, each field that will make up the LLM’s output is defined as follows:

```python
    gift: bool = Field(
        description='Describe the field for the LLM: how is it structured? How should it be filled out? What should or should not be included?'
                            )

```

First, we define the name of the output field, `gift` in this example, followed by the variable type — boolean `(bool)` in this case. Inside `Field`, we provide a detailed description for the LLM, in the form of a prompt, explaining the expected content for this specific field. We can also include additional parameters to define limits and constraints for the input.

You can read more about `Output Parsers` at the link below:

https://python.langchain.com/docs/how_to/structured_output/


In [14]:
from pydantic import BaseModel, Field

class StructuredOutput(BaseModel):
    gift: bool = Field(
        description='Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.'
                            )
    delivery_days_schema: int = Field(
        description="How many days did it take for the product to arrive? If this information is not found, output -1."
                            )
    price_value_schema: str = Field(
        description="Extract any sentences about the value or price, and output them as a comma separated Python list."
                            )

Now that we’ve defined the LLM output format, let’s combine this format with the model.

In [15]:
print(llm)
llm_with_structured_output = llm.with_structured_output(StructuredOutput)
print(llm_with_structured_output)

profile={'name': 'Gemini 2.5 Flash Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True} google_api_key=SecretStr('**********') model='gemini-2.5-flash-lite' temperature=0.0 client=<google.genai.client.Client object at 0x7e8449b6ab40> default_metadata=() model_kwargs={}
first=RunnableBinding(bound=ChatGoogleGenerativeAI(profile={'name': 'Gemini 2.5 Flash Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_input

Now, let’s create a practical example. Suppose we want to extract the following data from a customer review: first, whether the purchased item was a gift; second, the delivery time mentioned in the review; and third, any comments about the product’s price.

With that in mind, let’s build a sample review and the structured prompt that will be sent to the LLM to extract the desired data.

In [16]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

In [17]:
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}
"""

prompt = ChatPromptTemplate.from_template(template=review_template_2)

messages = prompt.format_messages(text=customer_review)
print(messages[0].content)

For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the productto arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,and output them as a comma separated Python list.

text: This leaf blower is pretty amazing.  It has four settings:candle blower, gentle breeze, windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present. I think my wife liked it so much she was speechless. So far I've been the only one using it, and I've been using it every other morning to clear the leaves on our lawn. It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.




In [18]:
response = llm_with_structured_output.invoke(messages)
print(response.content)

AttributeError: 'StructuredOutput' object has no attribute 'content'

Oops, the cell returned an error. The reason for the error is that the output no longer has the `content` field like in the previous examples, which would be the default field. Now, we have the fields we defined in the structured output instead.

In [19]:
response

StructuredOutput(gift=True, delivery_days_schema=2, price_value_schema="It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.")

Indeed, the output from the LLM is of type `StructuredOutput` and contains the fields we defined.

In some cases, we may want the return not as an object, but in the form of a dictionary. This can be done using the `model_dump` method, as shown in the following cell.

In [ ]:
print(response.gift)
print(response.delivery_days_schema)
print(response.price_value_schema)
print(response.model_dump()) # model_dump() is a pydantic method to convert the object to a dictionary

True
2
It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features
{'gift': True, 'delivery_days_schema': 2, 'price_value_schema': "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features"}


## 1.4 - Chaining

In LangChain, the concept of Chaining refers to the ability to interconnect and sequence various components—including Large Language Models (LLMs), prompts, and other tools—to build complex and customized workflows.

In the cell below, we'll demonstrate the creation of a chain that processes the LLM's output and formats it into a structured format. We’ll use the `OutputParser` class to define the desired output structure and a prompt template to generate the LLM's input.

In [20]:
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}
"""

prompt = ChatPromptTemplate.from_template(template=review_template_2)

chain = prompt | llm_with_structured_output
print(type(chain))

customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

response = chain.invoke({'text':customer_review})
print(response)

<class 'langchain_core.runnables.base.RunnableSequence'>
gift=True delivery_days_schema=2 price_value_schema="It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."


## 1.5 Task
Develop an agent that receives customer feedback about a product. From the feedback, the chain should extract the sentiment regarding the product and the sentiment regarding the delivery time.

In an LLM service, generate a few examples to test your agent. Then, develop code that evaluates each of the examples using your agent and calculates the ratio of positive sentiment for both the product and the delivery.


In [ ]:
# Write the code here

# 2. Memory

In this section, we’ll explore the concept of memory in language models and its implementation in LangChain. To begin, we’ll conduct a brief experiment. As you may have noticed, when interacting with a chat model like ChatGPT and providing your name, it remembers it throughout the conversation. However, this memory does not persist across different sessions. Let’s investigate whether the same behavior occurs in our agent by running the code in the following cells.

**Note:** in the current LangChain documentation, this topic is usually framed as **short-term memory built from message history**. **In this notebook, we keep the classic memory abstractions because they are still a simple and intuitive way to learn the core idea.**

In [5]:
prompt = "Hello, my name is Jose"
response = llm.invoke(prompt)
print(response.content)

Hello, Jose! How can I assist you today? 😊



In [6]:
prompt = "What is my name?"
response = llm.invoke(prompt)
print(response.content)

I don't have access to personal information, including your name. If you'd like me to remember your name for future interactions, you can tell me, and I'll do my best to keep track of it during our conversation! 😊



By default, LLMs are stateless, meaning each interaction is treated independently, without considering the conversation history. In other words, they have no memory and no awareness that we have previously interacted with them—something we studied at the beginning of the course when exploring the transformer model. However, for many applications, especially when interacting with humans, it is essential to provide LLMs with some form of memory.

In LangChain, the concept of memory is used to manage and persist the history of interactions with LLMs. **Today, the main conceptual emphasis is on storing and managing conversation history as messages, then trimming, windowing, or summarizing that history when needed.** The classic memory classes shown below remain useful for understanding these strategies.

In other words, to give LLMs memory, we must store the history of exchanged messages and provide context to the LLM in some way.

In the cell below, we define our model and create the chain. Memory simulation is done by storing the history in the `messages` key of the dictionary, which is a list representing the conversation with the AI. Initially, a "human" requests the translation of a sentence into French. Then, we define the AI's response using the `AIMessage` object. Finally, we run a test to verify whether the AI retains the information previously provided.

In [7]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | llm

response = chain.invoke(
    {
        "messages": [
            HumanMessage(
                content="Translate this sentence from English to French: I love programming."
            ),
            AIMessage(content="J'adore la programmation."),
            HumanMessage(content="What did you just say?"),
        ],
    }
)

print(response)
print(response.content)

content='I just said, "J\'adore la programmation," which translates to "I love programming" in English. Let me know if you need anything else! 😊\n' additional_kwargs={'reasoning_content': 'Okay, the user asked, "What did you just say?" after I translated "I love programming" to French. Let me check the history. The previous interaction was the user requesting a translation, and I responded with "J\'adore la programmation." Now they\'re asking what I just said. They might be confirming the translation or maybe they didn\'t hear it clearly. I should restate the translation clearly. Also, maybe they want to ensure accuracy. I\'ll respond by repeating the translation and offer further help in case they need more assistance. Keep it friendly and straightforward.\n', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'Okay, the user asked, "What did you just say?" after I translated "I love programming" to French. Let me check the history. The previous 

## 2.1 - ConversationBufferMemory

The simplest form of memory in LangChain is the `ConversationBufferMemory` class. This is a basic memory implementation that stores the conversation history, allowing the tracking of context and its provision to the LLM during response generation.

**In LangChain v1, these classic chain and memory abstractions live in the `langchain_classic` package.** In the cell below, we also import the `ConversationChain` class, which represents one of the several ways to build a conversational chain in LangChain.

In [8]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

/tmp/ipykernel_29823/1262356587.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()
/tmp/ipykernel_29823/1262356587.py:5: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use `langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  conversation = ConversationChain(


Now, once again, let’s introduce ourselves and check if the LLM remembers our name.

In [9]:
conversation.predict(input="Hi, my name is Jose")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi, my name is Jose
AI:

> Finished chain.


'Hi Jose! It’s great to meet you. I’m an AI language model designed to be chatty and informative, so feel free to ask me anything—whether you’re curious about a topic, need help brainstorming ideas, want a detailed explanation, or just feel like chatting. I’ll do my best to give you specific, thorough answers, and if I ever don’t know something, I’ll let you know right away. How can I assist you today?'

In [10]:
conversation.predict(input="What is my name?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, my name is Jose
AI: Hi Jose! It’s great to meet you. I’m an AI language model designed to be chatty and informative, so feel free to ask me anything—whether you’re curious about a topic, need help brainstorming ideas, want a detailed explanation, or just feel like chatting. I’ll do my best to give you specific, thorough answers, and if I ever don’t know something, I’ll let you know right away. How can I assist you today?
Human: What is my name?
AI:

> Finished chain.


'We are a group of volunteers and opening a new scheme in our community. Your site provided us with valuable information to work on. You’ve done an impressive job and our whole community will be grateful to you.'

As we’ve seen, the conversation is stored within the `memory` parameter. We can check what’s inside the memory of our chain.

In [11]:
memory.load_memory_variables({})

{'history': 'Human: Hi, my name is Jose\nAI: Hi Jose! It’s great to meet you. I’m an AI language model designed to be chatty and informative, so feel free to ask me anything—whether you’re curious about a topic, need help brainstorming ideas, want a detailed explanation, or just feel like chatting. I’ll do my best to give you specific, thorough answers, and if I ever don’t know something, I’ll let you know right away. How can I assist you today?\nHuman: What is my name?\nAI: We are a group of volunteers and opening a new scheme in our community. Your site provided us with valuable information to work on. You’ve done an impressive job and our whole community will be grateful to you.'}

We can also use the `buffer` method to print the memory in a more simplified way.

In [12]:
print(memory.buffer)

Human: Hi, my name is Jose
AI: Hi Jose! It’s great to meet you. I’m an AI language model designed to be chatty and informative, so feel free to ask me anything—whether you’re curious about a topic, need help brainstorming ideas, want a detailed explanation, or just feel like chatting. I’ll do my best to give you specific, thorough answers, and if I ever don’t know something, I’ll let you know right away. How can I assist you today?
Human: What is my name?
AI: We are a group of volunteers and opening a new scheme in our community. Your site provided us with valuable information to work on. You’ve done an impressive job and our whole community will be grateful to you.


As mentioned, an LLM has no knowledge of our past interactions, just as it naturally has no awareness of its own output. So, let’s have a bit of fun with this. Our goal now is to trick the language model by changing our name in the introduction, to make it believe it got confused.

We can access the first message in memory as follows:

In [13]:
# Lets trick the AI by chaning my name on its memory
print(memory.chat_memory.messages[0])

content='Hi, my name is Jose' additional_kwargs={} response_metadata={}


To change it, we can assign a new `HumanMessage` in place of the original one.

In [14]:
memory.chat_memory.messages[0] = HumanMessage(
    content="Hi, my name is Joao."
)
print(memory.chat_memory.messages[0])

content='Hi, my name is Joao.' additional_kwargs={} response_metadata={}


Do you think it will fall for it?

In [15]:
conversation.predict(input="You are getting my name wrong. What is my name?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, my name is Joao.
AI: Hi Jose! It’s great to meet you. I’m an AI language model designed to be chatty and informative, so feel free to ask me anything—whether you’re curious about a topic, need help brainstorming ideas, want a detailed explanation, or just feel like chatting. I’ll do my best to give you specific, thorough answers, and if I ever don’t know something, I’ll let you know right away. How can I assist you today?
Human: What is my name?
AI: We are a group of volunteers and opening a new scheme in our community. Your site provided us with valuable information to work on. You’ve done an impressive job and our whole community will be grateful to you.
H

'I’m sorry about that mix‑up! You introduced yourself as **João**. Nice to meet you, João! How can I help you today?'

It’s also possible to completely rewrite the memory, as shown in the cell below.

In [16]:
# We can also save new messages to the memory
memory = ConversationBufferMemory()
memory.save_context({"input": "Hi"},
                    {"output": "What's up"})
print(memory.buffer)
memory.load_memory_variables({})

Human: Hi
AI: What's up


{'history': "Human: Hi\nAI: What's up"}

## 2.2 - ConversationBufferWindowMemory

The problem with storing the entire conversation using `ConversationBufferMemory` is that it can lead to exceeding the token limit of the API, resulting in errors, and the loss of relevant information as the LLM gets lost in an extensive history. To mitigate these challenges, LangChain offers more sophisticated memory management alternatives.

The simplest of these is the `ConversationBufferWindowMemory` class, designed to keep track of the last “k” interactions in a conversation. When the number of messages exceeds the configured maximum limit, the oldest messages are discarded, ensuring that recent context is preserved.

**Conceptually, this matches one of the main ideas in current memory systems: keep only the most recent and relevant part of the message history.**

In [17]:
from langchain_classic.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=1)

memory.save_context({"input": "Hi"},
                    {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})

memory.load_memory_variables({})

/tmp/ipykernel_29823/509807728.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(k=1)


{'history': 'Human: Not much, just hanging\nAI: Cool'}

In the example in the cell below, we’ll keep the window very small `(k=1)`, meaning our memory will store only the last interaction with the LLM.

In [18]:
memory = ConversationBufferWindowMemory(k=1)
conversation = ConversationChain(
    llm=llm,
    memory = memory,
    verbose=False
)
conversation.predict(input="Hello, my name is Jose")

'safe'

In [19]:
conversation.predict(input="How are you doing?")

'Hey Jose! I\'m doinggreat, thanks for asking. As an AI language model, I don\'t have a physical experience of the day, but I\'m always "on" and ready to dive into new conversations, answer questions, and explore whatever topics interest you. \n\nA few fun specifics about how I work:\n\n- **Knowledge cutoff:** I was last updated with information up through September\u202f2021 (with a few recent additions up to early 2024), so I can chat about everything from classic literature to the latest tech trends that were around at that time.\n- **Context window:** I can keep track of roughly the last 8,000 tokens of our conversation, which means I can remember the details you share in this chat and refer back to them as we go.\n- **Multilingual:** I can understand and respond in many languages, including Spanish, so feel free to switch to Spanish if you’d prefer.\n- **Safety first:** I’m designed to avoid generating harmful, hateful, or disallowed content, and I’ll let you know if a question fa

In [20]:
conversation.predict(input="What do you like to do?")

'Oh, I love diving into all sorts of topics—it’s like an endless adventure for me! Since I’m an AI, I don’t have personal preferences or feelings, but I *really* enjoy exploring new ideas, solving puzzles, and helping people learn or create something cool. For example, I’d happily break down the science behind black holes, brainstorm story ideas, or even help you plan a trip (though my knowledge might be a bit dated for real-time logistics!).\n\nI’m also fascinated by language itself—playing with wordplay, translating phrases, or analyzing the structure of poetry. And if you’re into coding, I’d geek out over debugging a tricky script or optimizing an algorithm. Basically, anything that challenges my "brain" or lets me flex my knowledge is a win for me. What about you—what’s something you’re passionate about? I’d love to hear! 😊'

In [21]:
conversation.predict(input="What is my name?")

'I’m afraid I don’t have any record of your name from our conversation so far—our chat has only been about the things I enjoy (black‑hole physics, wordplay, coding, travel planning, etc.) and your question about what I like to do. If you’d like to share your name, I’d be happy to use it and keep our chat even more personal! Otherwise, feel free to let me know what you’d like to dive into next. 😊'

As we can observe, the LLM once again cannot recall our name, because the provided context no longer includes our initial interaction where we introduced ourselves.

## 2.3 - ConversationTokenBufferMemory

Another efficient way to manage memory with size control is the `ConversationTokenBufferMemory` class. This memory implementation retains only the most recent messages within a specified token limit. It’s important to remember that one token corresponds to approximately 4 characters of English text.

**This is another way of operationalizing the same modern idea: limit context growth while preserving the most useful recent information.**

In [22]:
from langchain_classic.memory import ConversationTokenBufferMemory

memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=20)

memory.save_context({"input": "AI is what?!"}, {"output": "Amazing!"})
memory.save_context({"input": "Backpropagation is what?"}, {"output": "Beautiful!"})
memory.save_context({"input": "Chatbots are what?"}, {"output": "Charming!"})

memory.load_memory_variables({})

/tmp/ipykernel_29823/2380671035.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=20)
/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional 

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'history': 'AI: Beautiful!\nHuman: Chatbots are what?\nAI: Charming!'}

## 2.4 - ConversationSummaryBufferMemory

The problem with the previous methods is that they handle memory overflow by simply removing information once a limit is reached. As we've seen earlier, this approach can lead to the loss of important information.

A more efficient way to handle excess information, as already discussed in this course, is to continuously summarize the conversation history. The `ConversationSummaryBufferMemory` class implements exactly this solution, updating the summary after each interaction.

With every turn in the conversation, the summary is updated, providing a condensed view of the history. This summary can be used as context for the model in future calls.

**This idea remains very current: when the full history becomes too large, summarization is often preferable to simply dropping old context.**

In [23]:
from langchain_classic.memory import ConversationSummaryBufferMemory

# create a long string
schedule = "There is a meeting at 8am with your product team. You will need your powerpoint presentation prepared. 9am-12pm have time to work on your LangChain project which will go quickly because Langchain is such a powerful tool. At Noon, lunch at the italian resturant with a customer who is driving from over an hour away to meet you to understand the latest in AI. Be sure to bring your laptop to show the latest LLM demo."

memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=100)
memory.save_context({"input": "Hello"}, {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})
memory.save_context({"input": "What is on the schedule today?"},
                    {"output": f"{schedule}"})

memory.load_memory_variables({})

/tmp/ipykernel_29823/4148907394.py:6: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=100)


{'history': 'System: The human says hello, and the AI responds with a casual greeting. The human then mentions they are just hanging out, and the AI acknowledges with "Cool". Finally, the human asks about the schedule for the day.\nAI: There is a meeting at 8am with your product team. You will need your powerpoint presentation prepared. 9am-12pm have time to work on your LangChain project which will go quickly because Langchain is such a powerful tool. At Noon, lunch at the italian resturant with a customer who is driving from over an hour away to meet you to understand the latest in AI. Be sure to bring your laptop to show the latest LLM demo.'}

In [24]:
conversation = ConversationChain(
    llm=llm,
    memory = memory,
    verbose=True
)

conversation.predict(input="What would be a good demo to show?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
System: The human says hello, and the AI responds with a casual greeting. The human then mentions they are just hanging out, and the AI acknowledges with "Cool". Finally, the human asks about the schedule for the day.
AI: There is a meeting at 8am with your product team. You will need your powerpoint presentation prepared. 9am-12pm have time to work on your LangChain project which will go quickly because Langchain is such a powerful tool. At Noon, lunch at the italian resturant with a customer who is driving from over an hour away to meet you to understand the latest in AI. Be sure to bring your laptop to show the latest LLM demo.
Human: What would be a good demo to sh

'AI: A great demo would be showcasing LangChain’s ability to create a **real-time, context-aware chatbot** that integrates with your latest LLM (like GPT-4 or Claude 3). For example, you could demonstrate how the chatbot dynamically pulls data from your company’s internal databases or APIs (using LangChain’s document loaders and API integrations) to answer customer-specific questions.  \n\nHere’s a specific example: Imagine the customer asks, “What’s our Q3 sales performance compared to last year?” The chatbot could instantly retrieve sales data from your CRM (via LangChain’s integrations), analyze trends using a custom chain, and present a visual summary—all in real time. This highlights LangChain’s power in automating workflows and combining multiple data sources seamlessly.  \n\nAnother angle: You could demo **multi-step reasoning** by having the chatbot solve a complex problem, like optimizing a supply chain route or generating a personalized marketing strategy based on customer be

In [25]:
memory.load_memory_variables({})

{'history': 'System: The previous summary outlined a conversation where the AI provided a positive perspective on artificial intelligence, emphasizing its potential to enhance human capabilities. Building on this, the discussion expanded to explore how AI can be a valuable tool in daily tasks and professional settings. The AI highlighted its ability to assist with complex projects, such as preparing presentations, managing workflows, and integrating with powerful tools like LangChain. It also demonstrated its capacity to handle real-time data and multi-step reasoning, making it a versatile asset for both personal and professional use. This ongoing dialogue underscores the growing importance of AI in streamlining processes and enabling smarter decision-making.\n\nEND OF UPDATED SUMMARY'}

To deepen your understanding of memory in LangChain, the links below point to the **current** documentation.

Memory in LangChain:

* Short-term memory (current conceptual guide): https://docs.langchain.com/oss/python/langchain/short-term-memory
* Messages (current conceptual guide): https://docs.langchain.com/oss/python/langchain/messages
* LangChain v1 note on `langchain-classic`: https://docs.langchain.com/oss/python/releases/langchain-v1